In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV

In [7]:
decision_tree = DecisionTreeClassifier()
random_forest = RandomForestClassifier()

params_tree = [{'criterion' : ['gini', 'entropy'], 'max_depth' : [2, 5, 10, 15], 'min_samples_split' : [2, 5, 10, 15],
                'min_samples_leaf' : [2, 5, 10, 15]}]
params_forest = [{'n_estimators' : [100, 200], 'max_depth' : [None, 5, 10, 15, 20], 'min_samples_split' : [2, 5, 10],
                  'min_samples_leaf' : [1, 2, 5, 10], 'bootstrap' : [True, False]}]

search_tree = RandomizedSearchCV(decision_tree, params_tree, cv = 5, n_iter = 100)
search_forest = RandomizedSearchCV(random_forest, params_forest, cv = 5, n_iter = 30)

## Dataset 1: Instacart Dataset

In [8]:
data1 = pd.read_csv('https://raw.githubusercontent.com/eghatzis0527/DX699---HW-Assignments/refs/heads/main/instacart.csv')

transform_cols = ['times_purchased', 'num_orders', 'frequency', 'Product Popularity', 'Last Product Order', 'Last Order', 'Orders Since Last Purchase']

for col in transform_cols:
    data1[f'log_{col}'] = np.log1p(data1[col])

data1_transformed = data1.drop(columns = ['times_purchased', 'num_orders', 'frequency', 'Product Popularity', 'Last Product Order', 'Last Order', 'Orders Since Last Purchase'])

data1_transformed.head()

,user_id,product_id,ordered,log_times_purchased,log_num_orders,log_frequency,log_Product Popularity,log_Last Product Order,log_Last Order,log_Orders Since Last Purchase
0,71,45,0.0,1.791759,3.178054,0.196710,0.005878,2.302585,3.178054,2.708050
1,71,117,1.0,2.995732,3.178054,0.602175,0.001219,3.178054,3.178054,0.000000
2,71,2078,0.0,0.693147,3.178054,0.042560,0.006478,1.386294,3.178054,3.044522
3,71,2825,0.0,1.098612,3.178054,0.083382,0.003743,1.945910,3.178054,2.890372
4,71,3376,1.0,1.098612,3.178054,0.083382,0.003602,2.995732,3.178054,1.609438


In [ ]:
X = data1_transformed.drop(columns = ['ordered', 'user_id', 'product_id'])
y = data1_transformed['ordered']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 42)

search_tree.fit(X_train, y_train)
print(f'Best Hyperparameters for Decision Tree: {search_tree.best_params_}')

search_forest.fit(X_train, y_train)
print(f'Best Hyperparameters for Random Forest: {search_forest.best_params_}')

Best Hyperparameters for Decision Tree: {'min_samples_split': 10, 'min_samples_leaf': 15, 'max_depth': 5, 'criterion': 'gini'}


In [ ]:
best_tree = DecisionTreeClassifier()

best_tree.fit(X_train, y_train)

y_preds = best_tree.predict_proba()[:, 1]

values = pd.DataFrame({'Predicted' : y_preds, 'Actual' : y_test})
values['decile'] = pd.qcut(values['Predicted'], q = 10, labels = False) + 1

lift_table = values.groupby('decile').agg(avg_predicted = ('Predicted', 'mean'), avg_ordered = ('Actual', 'mean'))

overall_ordered = y_test.mean()

lift_table['lift'] = lift_table['avg_ordered']/overall_ordered

lift_table['avg_predicted'] = lift_table['avg_predicted'].map(lambda x: f'{x:.3f}')
lift_table['avg_ordered'] = lift_table['avg_ordered'].map(lambda x: f'{x:.3f}')
lift_table['lift'] = lift_table['lift'].map(lambda x: f'{x:.3f}')

In [ ]:
best_forest = RandomForestClassifier()

best_forest.fit(X_train, y_train)

y_preds = best_forest.predict_proba()[:, 1]

values = pd.DataFrame({'Predicted' : y_preds, 'Actual' : y_test})
values['decile'] = pd.qcut(values['Predicted'], q = 10, labels = False) + 1

lift_table = values.groupby('decile').agg(avg_predicted = ('Predicted', 'mean'), avg_ordered = ('Actual', 'mean'))

overall_ordered = y_test.mean()

lift_table['lift'] = lift_table['avg_ordered']/overall_ordered

lift_table['avg_predicted'] = lift_table['avg_predicted'].map(lambda x: f'{x:.3f}')
lift_table['avg_ordered'] = lift_table['avg_ordered'].map(lambda x: f'{x:.3f}')
lift_table['lift'] = lift_table['lift'].map(lambda x: f'{x:.3f}')

## Dataset 2: Amazon Reviews Dataset

In [ ]:
import ast
from sklearn.preprocessing import MultiLabelBinarizer

small_reviews = pd.read_csv('https://raw.githubusercontent.com/eghatzis0527/DX699---HW-Assignments/refs/heads/main/small_reviews%5B1%5D.csv')

small_meta = pd.read_csv('https://raw.githubusercontent.com/eghatzis0527/DX699---HW-Assignments/refs/heads/main/small_meta%5B1%5D.csv')

small_meta = small_meta.set_index(keys = 'parent_asin')

small_meta = small_meta.drop(columns = ['title', 'images'])

data2 = small_reviews.join(small_meta, on = 'parent_asin', how = 'inner')

data2 = data2.drop(columns = ['main_category', 'store', 'rating_number', 'videos', 'images', 'subtitle', 'author', 'bought_together', 'features', 'description', 'details', 'timestamp', 'text', 'title'])

data2['product_review_count'] = data2.groupby('asin').transform('size')
data2['user_review_count'] = data2.groupby('user_id').transform('size')

data2['price'] = data2['price'].fillna(data2['price'].median())

data2['categories'] = data2['categories'].apply(ast.literal_eval)

mlb = MultiLabelBinarizer()

categories = pd.DataFrame(mlb.fit_transform(data2['categories']), columns = mlb.classes_, index = data2.index)

top_30 = categories.sum().sort_values(ascending = False).head(31).index

categories = categories[top_30[1:31]]

data2 = data2.drop(columns = 'categories')

data2 = data2.join(categories)

columns_to_transform = ['helpful_vote', 'average_rating', 'price', 'product_review_count', 'user_review_count']

for col in columns_to_transform:
    data2[f'log_{col}'] = np.log1p(data2[col])

data2_transformed = data2.drop(columns = ['helpful_vote', 'average_rating', 'price', 'product_review_count', 'user_review_count'])

data2_transformed = data2_transformed.reset_index(drop = True)

for i in range(len(data2_transformed)):
    if data2_transformed.iloc[i]['rating'] == 4:
        data2_transformed.loc[i, 'liked'] = 1
    elif data2_transformed.iloc[i]['rating'] == 5:
        data2_transformed.loc[i, 'liked'] = 1
    else:
        data2_transformed.loc[i, 'liked'] = 0

X = data2_transformed.drop(columns = ['asin', 'parent_asin', 'user_id', 'liked'])
y = data2_transformed['liked']

X.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.25, random_state = 42)

search_tree.fit(X_train, y_train)
print(f'Best Hyperparameters for Decision Tree: {search_tree.best_params_}')

search_forest.fit(X_train, y_train)
print(f'Best Hyperparameters for Random Forest: {search_forest.best_params_}')

In [ ]:
best_tree = DecisionTreeClassifier()

best_tree.fit(X_train, y_train)

y_preds = best_tree.predict_proba()[:, 1]

values = pd.DataFrame({'Predicted' : y_preds, 'Actual' : y_test})
values['decile'] = pd.qcut(values['Predicted'], q = 10, labels = False) + 1

lift_table = values.groupby('decile').agg(avg_predicted = ('Predicted', 'mean'), avg_liked = ('Actual', 'mean'))

overall_liked = y_test.mean()

lift_table['lift'] = lift_table['avg_liked']/overall_liked

lift_table['avg_predicted'] = lift_table['avg_predicted'].map(lambda x: f'{x:.3f}')
lift_table['avg_liked'] = lift_table['avg_liked'].map(lambda x: f'{x:.3f}')
lift_table['lift'] = lift_table['lift'].map(lambda x: f'{x:.3f}')

In [ ]:
best_forest = RandomForestClassifier()

best_forest.fit(X_train, y_train)

y_preds = best_forest.predict_proba()[:, 1]

values = pd.DataFrame({'Predicted' : y_preds, 'Actual' : y_test})
values['decile'] = pd.qcut(values['Predicted'], q = 10, labels = False) + 1

lift_table = values.groupby('decile').agg(avg_predicted = ('Predicted', 'mean'), avg_liked = ('Actual', 'mean'))

overall_liked = y_test.mean()

lift_table['lift'] = lift_table['avg_liked']/overall_liked

lift_table['avg_predicted'] = lift_table['avg_predicted'].map(lambda x: f'{x:.3f}')
lift_table['avg_liked'] = lift_table['avg_liked'].map(lambda x: f'{x:.3f}')
lift_table['lift'] = lift_table['lift'].map(lambda x: f'{x:.3f}')

## Dataset 3: Amazon Reviews Dataset